# Covariance matrix estimation: methods compared

Given daily returns for a set of tickers, several ways to estimate the NxN
covariance, and a like-for-like comparison. With N names and T observations the
sample covariance has N(N+1)/2 parameters; when N approaches T it is
ill-conditioned (largest eigenvalues overstated, smallest understated) and a
minimum-variance optimiser loads on the noisiest directions. Every method below
fights that estimation error a different way.

Methods: sample, Ledoit-Wolf linear shrinkage, OAS, Ledoit-Wolf nonlinear
shrinkage (QIS), a PCA statistical factor model, random-matrix eigenvalue
clipping, EWMA (RiskMetrics), and a GARCH-based conditional covariance (constant
conditional correlation). DCC is shown at the end as the time-varying-correlation
extension.

Most of these keep the sample eigenvectors and only change the eigenvalues, so
the middle section shows the shrinkage as a curve on the spectrum: what each one
does to a sample eigenvalue. The final test is economic: build the global
minimum-variance portfolio from each estimate out-of-sample and measure its
realised volatility. Lower is better.

In [ ]:
import warnings
import numpy as np
import polars as pl
import duckdb
import plotly.express as px
import plotly.graph_objects as go
from sklearn.covariance import LedoitWolf, OAS
from joblib import Parallel, delayed
from arch import arch_model
from scipy.optimize import minimize

from sdp.config import settings

warnings.filterwarnings("ignore")   # quiet GARCH convergence chatter
wh = duckdb.connect(str(settings.warehouse_path), read_only=True)

## The return panel

Liquid common stock only (`in_universe` excludes ETFs, which are linear combinations of their constituents and would let the optimiser build near-riskless arbitrage pairs). Names with a complete history over the window, so every estimator sees the same T x N matrix.

In [ ]:
N_NAMES = 2000      # tickers
TOTAL = 780        # trailing sessions of prices

latest = wh.execute("select max(date) from main_staging.stg_universe").fetchone()[0]
names = wh.execute(f"""
    select ticker from main_staging.stg_universe
    where date = date '{latest}' and in_universe
    order by adv desc limit {N_NAMES * 2}
""").pl()["ticker"].to_list()
tickstr = "('" + "','".join(names) + "')"

px_wide = (wh.sql(f"""
    select ticker, date, adj_close_total
    from main_staging.stg_prices_adjusted
    where adj_close_total is not null and ticker in {tickstr}
    order by date
""").pl()
    .pivot(values="adj_close_total", index="date", on="ticker")
    .sort("date").tail(TOTAL))

# Keep only fully-populated columns, then the first N_NAMES.
px_wide = px_wide[:, [c for c in px_wide.columns if c == "date" or px_wide[c].null_count() == 0]]
tickers = [c for c in px_wide.columns if c != "date"][:N_NAMES]
R = np.diff(np.log(px_wide.select(tickers).to_numpy()), axis=0)   # T x N log returns
T, N = R.shape
print(f"T={T} sessions, N={N} names, q=N/T={N/T:.2f}")
print(f"mean single-name vol {R.std(0).mean()*np.sqrt(252)*100:.1f}%/yr, "
      f"equal-weight {R.mean(1).std()*np.sqrt(252)*100:.1f}%/yr")

## The estimators

Each takes a T x N return matrix and returns an N x N covariance. Shrinkage and
OAS come from scikit-learn; PCA, RMT clipping, EWMA and the GARCH model are built
here so the mechanics are visible.

In [ ]:
def gmv(S):
    """Global minimum-variance weights, w = S^-1 1 / (1' S^-1 1)."""
    n = S.shape[0]; one = np.ones(n)
    try:
        x = np.linalg.solve(S, one)
    except np.linalg.LinAlgError:
        x = np.linalg.solve(S + 1e-8 * np.trace(S) / n * np.eye(n), one)
    return x / (one @ x)


def c_sample(X):
    return np.cov(X, rowvar=False)


def c_lw(X):
    return LedoitWolf().fit(X).covariance_        # linear shrinkage to scaled identity


def c_oas(X):
    return OAS().fit(X).covariance_               # oracle-approximating shrinkage


def nonlin_shrink(X, return_spectrum=False):
    """Ledoit-Wolf nonlinear shrinkage, the quadratic-inverse shrinkage (QIS) form.

    Linear shrinkage pulls every eigenvalue toward the mean by one intensity.
    Nonlinear shrinkage gives each sample eigenvalue its own optimal correction
    under Frobenius loss, computed in inverse-eigenvalue space from a smoothed
    estimate of the spectral density and its Hilbert transform. It keeps the
    sample eigenvectors and only cleans the eigenvalues, so it is the optimal
    rotation-invariant estimator. Works for both T>N and N>T; the null space of
    an N>T sample gets one common floor. With return_spectrum it also gives the
    sample eigenvalues and their shrunk images, both ascending, for the shrinkage
    curve. Port of the authors' reference (Ledoit-Wolf, BSD-2)."""
    T, N = X.shape
    Xc = X - X.mean(0)
    n = T - 1                                     # effective size after demeaning
    c = N / n                                     # concentration ratio N/T
    S = (Xc.T @ Xc) / n; S = (S + S.T) / 2
    lam, u = np.linalg.eigh(S)                    # ascending
    lam = np.clip(lam, 0.0, None)
    h = (min(c ** 2, 1 / c ** 2) ** 0.35) / N ** 0.35      # bounded bandwidth
    inv = 1.0 / lam[max(1, N - n + 1) - 1:N]      # inverse of the non-null eigenvalues
    m = inv.size
    Lj = np.tile(inv, (m, 1)).T                   # Lj[i, j] = inv[i]
    Lji = Lj - Lj.T
    denom = Lji ** 2 + (Lj ** 2) * h ** 2
    theta = np.mean(Lj * Lji / denom, axis=0)     # smoothed Stein shrinker
    htheta = np.mean(Lj * Lj * h / denom, axis=0)  # its Hilbert conjugate
    atheta2 = theta ** 2 + htheta ** 2
    if N <= n:                                    # T>N: sample is invertible
        delta = 1.0 / ((1 - c) ** 2 * inv + 2 * c * (1 - c) * inv * theta
                       + c ** 2 * inv * atheta2)
    else:                                         # N>T: fill the null space with one floor
        delta0 = 1.0 / ((c - 1) * np.mean(inv))
        delta = np.concatenate([np.repeat(delta0, N - n), 1.0 / (inv * atheta2)])
    delta = delta * (lam.sum() / delta.sum())     # preserve trace
    if return_spectrum:
        return lam, delta
    return (u * delta) @ u.T


def c_nls(X):
    return nonlin_shrink(X)


def c_pca(X):
    """Statistical factor model: top-K eigen-directions + diagonal idiosyncratic.
    K is the count of eigenvalues above the Marchenko-Pastur bulk."""
    S = np.cov(X, rowvar=False)
    v, _ = np.linalg.eigh(np.corrcoef(X, rowvar=False))
    q = X.shape[1] / X.shape[0]; lam_plus = (1 + np.sqrt(q)) ** 2
    K = max(1, int((v > lam_plus).sum()))
    vs, Vs = np.linalg.eigh(S); idx = np.argsort(vs)[::-1][:K]
    low_rank = Vs[:, idx] @ np.diag(vs[idx]) @ Vs[:, idx].T
    idio = np.clip(np.diag(S) - np.diag(low_rank), 1e-12, None)
    return low_rank + np.diag(idio)


def c_rmt(X):
    """Random-matrix clipping: keep eigenvalues above the MP edge, replace the
    noisy bulk with their average, rebuild the correlation, rescale by vols."""
    S = np.cov(X, rowvar=False); C = np.corrcoef(X, rowvar=False)
    d = np.sqrt(np.diag(S)); v, V = np.linalg.eigh(C)
    q = X.shape[1] / X.shape[0]; lam_plus = (1 + np.sqrt(q)) ** 2
    keep = v > lam_plus
    v_clean = np.where(keep, v, v[~keep].mean() if (~keep).any() else 0.0)
    v_clean *= C.shape[0] / v_clean.sum()          # preserve trace
    Cc = V @ np.diag(v_clean) @ V.T
    dd = np.sqrt(np.diag(Cc)); Cc = Cc / np.outer(dd, dd)
    return np.outer(d, d) * Cc


def c_ewma(X, lam=0.94):
    """RiskMetrics exponentially weighted covariance. Reactive, but with ~1/(1-lam)
    effective observations it is rank-deficient for large N (bad for inversion)."""
    Xc = X - X.mean(0); t = len(Xc)
    w = (1 - lam) * lam ** np.arange(t)[::-1]; w /= w.sum()
    return (Xc * w[:, None]).T @ Xc


def _fit_col(col):
    """Univariate GARCH(1,1); return the standardised return series."""
    try:
        res = arch_model(col * 100, mean="Zero", vol="GARCH", p=1, q=1,
                         rescale=False).fit(disp="off", show_warning=False)
        return col / (res.conditional_volatility / 100)
    except Exception:
        return col / col.std()


def c_ccc_garch(X):
    """Constant conditional correlation: current GARCH vols on the diagonal, a
    shrunk correlation of the standardised residuals off it."""
    def _fit_both(col):
        try:
            res = arch_model(col * 100, mean="Zero", vol="GARCH", p=1, q=1,
                             rescale=False).fit(disp="off", show_warning=False)
            s = res.conditional_volatility / 100
            return s[-1], col / s
        except Exception:
            sd = col.std(); return sd, col / sd
    out = Parallel(n_jobs=-1)(delayed(_fit_both)(X[:, j]) for j in range(X.shape[1]))
    d_now = np.array([o[0] for o in out])
    Z = np.column_stack([o[1] for o in out])
    Rz = LedoitWolf().fit(Z).covariance_
    dz = np.sqrt(np.diag(Rz)); Rz = Rz / np.outer(dz, dz)
    D = np.diag(d_now)
    return D @ Rz @ D


ESTIMATORS = {"sample": c_sample, "ledoit_wolf": c_lw, "oas": c_oas, "nonlin_lw": c_nls,
              "pca": c_pca, "rmt": c_rmt, "ewma": c_ewma, "ccc_garch": c_ccc_garch}
FAST = {k: v for k, v in ESTIMATORS.items() if k != "ccc_garch"}

## Structure of each estimate

Condition number on the full sample: the sample matrix is orders of magnitude worse conditioned than the regularised ones.

In [ ]:
rows = []
for name, fn in FAST.items():
    S = fn(R)
    rows.append({"method": name, "condition_number": float(np.linalg.cond(S))})
pl.DataFrame(rows).sort("condition_number")

In [ ]:
# Eigenvalue spectra (log scale). Sample has a few huge eigenvalues and a long
# tail of tiny ones (near zero, so off the log axis, when N>T); the regularised
# methods lift the floor. Nonlinear shrinkage keeps the top and pulls the bottom.
fig = go.Figure()
for name in ("sample", "ledoit_wolf", "oas", "nonlin_lw", "pca", "rmt"):
    ev = np.sort(np.linalg.eigvalsh(ESTIMATORS[name](R)))[::-1]
    fig.add_trace(go.Scatter(y=ev, mode="lines", name=name))
fig.update_yaxes(type="log", title="eigenvalue (log)")
fig.update_layout(title="Covariance eigenvalue spectra (full sample)",
                  xaxis_title="rank", height=440)
fig.show()

## What the estimators do to the eigenvalues

Sample, both shrinkages, and RMT clipping keep the sample eigenvectors and only
change the eigenvalues. Each is then a curve that maps a raw eigenvalue to a
cleaned one. The reason a curve is needed is the Marchenko-Pastur result below:
pure noise spreads the eigenvalues into a band, so the largest sample
eigenvalues are biased up and the smallest biased down.

- **Linear shrinkage (LW, OAS)** is a straight line: pull every eigenvalue toward
  the mean by one intensity. OAS pulls a little harder.
- **Nonlinear shrinkage (QIS)** is a curve: leave the top (signal) almost alone,
  lift the bottom (noise) hard toward a floor. It is the best choice among all
  methods that keep the sample eigenvectors.
- **RMT clipping** is a step: keep the eigenvalues above the noise edge, flatten
  the whole bulk to one level.

In [ ]:
# Marchenko-Pastur: where pure noise puts the eigenvalues. Correlation
# eigenvalues inside the band [lam_minus, lam_plus] are consistent with noise;
# those above the edge are signal (real common factors). This is why a flat
# shrinkage is wrong and a curve is needed.
C = np.corrcoef(R, rowvar=False)
evc = np.sort(np.linalg.eigvalsh(C))[::-1]
qf = N / T
lam_m, lam_p = (1 - np.sqrt(qf)) ** 2, (1 + np.sqrt(qf)) ** 2
xs = np.linspace(max(lam_m, 1e-6), lam_p, 300)
mp = np.sqrt(np.clip((lam_p - xs) * (xs - lam_m), 0, None)) / (2 * np.pi * qf * xs)
n_sig = int((evc > lam_p).sum())
n_null = int((evc <= 1e-8).sum())
bulk = evc[(evc > 1e-8) & (evc <= lam_p * 1.5)]       # nonzero, near the band

fig = go.Figure()
fig.add_trace(go.Histogram(x=bulk, histnorm="probability density", nbinsx=80,
                           name="sample eigenvalues", marker_color="#8b949e"))
fig.add_trace(go.Scatter(x=xs, y=mp, mode="lines", name="Marchenko-Pastur (pure noise)",
                         line=dict(color="#cf222e", width=2)))
fig.add_vline(x=lam_p, line_dash="dash", line_color="#1a7f37",
              annotation_text=f"edge {lam_p:.2f}")
fig.update_layout(title=f"Correlation eigenvalues vs the MP noise band "
                        f"({n_sig} above the edge = signal, top = {evc[0]:.0f})",
                  xaxis_title="eigenvalue", yaxis_title="density",
                  xaxis_range=[0, lam_p * 1.5], height=420)
fig.show()
print(f"q=N/T={qf:.2f}   noise band [{lam_m:.2f}, {lam_p:.2f}]   "
      f"{n_sig} of {N} eigenvalues are signal, {n_null} are null (N>T); "
      f"market factor = {evc[0]:.0f}")

In [ ]:
# The shrinkage curve: raw sample eigenvalue -> cleaned eigenvalue, for the
# methods that keep the sample eigenvectors. Sample is the 45-degree line; linear
# shrinkage is a straight pull toward the mean; nonlinear shrinkage is a curve
# (top kept, bottom lifted); RMT clipping is a step at the noise edge.
S = np.cov(R, rowvar=False)
lam = np.sort(np.linalg.eigvalsh(S))                  # ascending
mu = lam.mean()
s_lw = LedoitWolf().fit(R).shrinkage_
s_oas = OAS().fit(R).shrinkage_
lin = (1 - s_lw) * lam + s_lw * mu
oas = (1 - s_oas) * lam + s_oas * mu
lam_nl, d_nl = nonlin_shrink(R, return_spectrum=True)
# RMT clip in covariance space, so it too is a function of the same lam.
edge = mu * (1 + np.sqrt(N / T)) ** 2
keep = lam > edge
step = np.where(keep, lam, lam[~keep].mean() if (~keep).any() else mu)
step *= lam.sum() / step.sum()

pos = lam > 0                                         # log axis: drop the null space
fig = go.Figure()
fig.add_trace(go.Scatter(x=lam[pos], y=lam[pos], mode="lines", name="sample (no change)",
                         line=dict(color="#8b949e", dash="dash")))
fig.add_trace(go.Scatter(x=lam[pos], y=lin[pos], mode="lines",
                         name=f"linear LW (delta={s_lw:.2f})", line=dict(color="#1f6feb")))
fig.add_trace(go.Scatter(x=lam[pos], y=oas[pos], mode="lines",
                         name=f"OAS (delta={s_oas:.2f})", line=dict(color="#8250df")))
fig.add_trace(go.Scatter(x=lam_nl[lam_nl > 0], y=d_nl[lam_nl > 0], mode="lines",
                         name="nonlinear LW (QIS)", line=dict(color="#cf222e", width=3)))
fig.add_trace(go.Scatter(x=lam[pos], y=step[pos], mode="lines", name="RMT clip",
                         line=dict(color="#1a7f37", shape="hv")))
fig.add_vline(x=edge, line_dash="dot", line_color="#1a7f37", annotation_text="MP edge")
fig.update_xaxes(type="log", title="sample eigenvalue (log)")
fig.update_yaxes(type="log", title="cleaned eigenvalue (log)")
fig.update_layout(title="What each method does to a sample eigenvalue", height=520)
fig.show()

In [ ]:
# Nonlinear shrinkage is well-posed in both regimes. Here on the full sample and
# on a short 126-day window: positive-definite and far better conditioned than
# the sample matrix in each, whether T>N or N>T.
def _regime(Xw, label):
    Tw, Nw = Xw.shape
    Snl = nonlin_shrink(Xw); Ssa = np.cov(Xw, rowvar=False)
    ev = np.linalg.eigvalsh(Snl)
    return {"regime": label, "T": Tw, "N": Nw, "q_NoverT": round(Nw / Tw, 2),
            "nls_min_eig": float(ev.min()), "nls_posdef": bool(ev.min() > 0),
            "nls_cond": round(float(np.linalg.cond(Snl)), 1),
            "sample_cond": float(np.linalg.cond(Ssa))}

pl.DataFrame([_regime(R, "full sample"),
              _regime(R[-126:], "126-day window")])

### Constructing the same estimator three ways

The shrinkage curve above is one object, the Ledoit-Peche oracle:

`d(lam) = lam / |1 - c + c*lam*g(lam + i0)|^2`,  with `c = N/T` and
`g(z) = (1/N) sum_j 1/(z - lam_j)` the Stieltjes transform of the sample spectrum.

Three routes reach it:

- **QIS** (used above) rewrites the oracle in inverse-eigenvalue space. Robust in
  both regimes, no tuning, no special functions. This is the one to use.
- **Epanechnikov kernel** (Ledoit-Wolf 2020) estimates the spectral density and
  its Hilbert transform. The kernel is chosen because its Hilbert transform is a
  closed form (a log), from a one-line principal-value integral.
- **Stieltjes / Cauchy** evaluates `g` a hair above the real axis. By
  Sokhotski-Plemelj the density and its Hilbert transform are the imaginary and
  real parts of that one complex sum, so no Hilbert transform is computed at all.
  The small imaginary part is the bandwidth.

**T>N and N>T are not the same problem.** When N>T the sample has N-T null
eigenvalues that carry probability mass `N_null/N` at zero. That mass belongs in
`g`; drop it and the non-null eigenvalues shrink too far. In the Cauchy form it is
one extra term (`nz / z`), so that construction stays correct as N/T crosses 1.
The Epanechnikov kernel needs the same correction plus a well-behaved density
estimate near zero, which it does not have, so it drifts from QIS when N>T. QIS
sidesteps both problems by working in inverse-eigenvalue space, which is why it is
the production estimator. The cell below prints the agreement for the current
`q = N/T`.

The cost of all three is the `O(N^2)` eigenvalue-pair sum. At N in the thousands
that is still milliseconds; past that, the sum is a kernel summation that a fast
multipole method or an FFT on a binned density turns near-linear.

In [ ]:
def _pos_eig(X):
    T, N = X.shape; Xc = X - X.mean(0); n = T - 1
    Sm = (Xc.T @ Xc) / n; Sm = (Sm + Sm.T) / 2
    lam, u = np.linalg.eigh(Sm); lam = np.clip(lam, 0, None)
    keep = lam > lam.max() * N * np.finfo(lam.dtype).eps
    return lam, u, keep, n, N / n


def nls_stieltjes(X, eta_scale=0.5):
    """Cauchy construction: g evaluated a hair above the real axis. The density
    and its Hilbert transform are the imaginary and real parts of one complex
    sum, so no Hilbert transform is computed. The imaginary part is the bandwidth.
    When N>T the N-T null eigenvalues carry mass N_null/N at zero, which must
    enter g (the `nz / z` term) or the non-null eigenvalues shrink too far. That
    is one line here, so this form is correct in both regimes."""
    lam, u, keep, n, c = _pos_eig(X); N = lam.size; lk = lam[keep]; nz = N - lk.size
    h = (min(c ** 2, 1 / c ** 2) ** 0.35) / N ** 0.35
    z = lk + 1j * eta_scale * h * lk
    g = (np.sum(1.0 / (z[:, None] - lk[None, :]), axis=1) + nz / z) / N   # null mass at 0
    d = np.empty(N)
    d[keep] = lk / np.abs(1 - c + c * lk * g) ** 2
    d[~keep] = 1.0 / ((c - 1) * np.mean(1 / lk)) if c > 1 else lk.min()
    return lam, d * (lam.sum() / d.sum())


r5 = np.sqrt(5.0)


def _Hk(x):
    """Hilbert transform of the Epanechnikov kernel, closed form from PV integration:
    H k(x) = 3x/(10 pi) + 3/(4 sqrt5 pi) (1 - x^2/5) log|(sqrt5 + x)/(sqrt5 - x)|."""
    with np.errstate(divide="ignore", invalid="ignore"):
        out = (3 * x / (10 * np.pi)
               + 3 / (4 * r5 * np.pi) * (1 - x ** 2 / 5) * np.log(np.abs((r5 + x) / (r5 - x))))
    return np.where(np.abs(x) == r5, 3 * x / (10 * np.pi), out)


def nls_epanechnikov(X):
    """Ledoit-Wolf 2020 analytical form: a variable-bandwidth Epanechnikov estimate
    of the spectral density f and its Hilbert transform, into the same oracle. The
    null mass enters the Hilbert part as N_null/(N*lam). Accurate for T>N; for N>T
    the kernel density near zero is fussy, so it drifts from QIS. That boundary
    trouble is exactly why the production estimator is the QIS inverse algebra."""
    lam, u, keep, n, c = _pos_eig(X); N = lam.size; lk = lam[keep]; nz = N - lk.size
    hb = n ** (-1 / 3) * lk[None, :]                    # local bandwidth lam_j * n^-1/3
    xg = (lk[:, None] - lk[None, :]) / hb
    f = np.mean(0.75 / r5 * np.maximum(1 - xg ** 2 / 5, 0) / hb, axis=1)
    G = np.mean(np.pi * _Hk(xg) / hb, axis=1) + (nz / N) / lk   # Hilbert transform + null mass
    g = G - 1j * np.pi * f
    d = np.empty(N)
    d[keep] = lk / np.abs(1 - c + c * lk * g) ** 2
    d[~keep] = 1.0 / ((c - 1) * np.mean(1 / lk)) if c > 1 else lk.min()
    return lam, d * (lam.sum() / d.sum())

In [ ]:
# All three reconstruct the same oracle. Overlay the curves on the full sample.
# Agreement is measured on the non-null eigenvalues (the ones the sample can see).
# T>N: all three coincide. N>T: the Cauchy form tracks QIS; the Epanechnikov
# kernel drifts, because its density estimate near zero is unreliable.
lam_q, d_q = nonlin_shrink(R, return_spectrum=True)
lam_s, d_s = nls_stieltjes(R)
lam_e, d_e = nls_epanechnikov(R)
p = lam_q > lam_q.max() * len(lam_q) * np.finfo(lam_q.dtype).eps    # non-null eigenvalues
fig = go.Figure()
fig.add_trace(go.Scatter(x=lam_q[p], y=lam_q[p], mode="lines", name="sample (no change)",
                         line=dict(color="#8b949e", dash="dash")))
fig.add_trace(go.Scatter(x=lam_q[p], y=d_q[p], mode="lines", name="QIS (inverse algebra)",
                         line=dict(color="#cf222e", width=3)))
fig.add_trace(go.Scatter(x=lam_e[p], y=d_e[p], mode="markers", name="Epanechnikov kernel",
                         marker=dict(color="#1f6feb", size=5)))
fig.add_trace(go.Scatter(x=lam_s[p], y=d_s[p], mode="markers", name="Stieltjes (Cauchy)",
                         marker=dict(color="#1a7f37", size=5, symbol="x")))
fig.update_xaxes(type="log", title="sample eigenvalue (log)")
fig.update_yaxes(type="log", title="cleaned eigenvalue (log)")
fig.update_layout(title=f"One oracle, three constructions (q=N/T={N / T:.2f})", height=480)
fig.show()
med = lambda d: float(np.median(np.abs(d[p] - d_q[p]) / d_q[p]))
regime = "T>N (all three coincide)" if N <= T else "N>T (Cauchy tracks QIS, kernel drifts)"
print(f"regime {regime}")
print(f"median |rel diff| vs QIS on non-null eigenvalues:  "
      f"Stieltjes {med(d_s):.1%}   Epanechnikov {med(d_e):.1%}")
print("Use QIS (nonlin_lw): correct and robust in both regimes, no tuning. The "
      "other two are here to show the oracle is one object.")

## Economic comparison: out-of-sample minimum-variance risk

At each rebalance, estimate the covariance on a trailing window, form the global
minimum-variance portfolio, and hold it forward. The realised volatility of the
concatenated out-of-sample returns is the score; gross leverage (sum of absolute
weights) shows how each estimator behaves. A short `EST_WIN` stresses the
estimators (q = N/EST_WIN is larger).

In [ ]:
def backtest(fn, win, step=21):
    oos, lev = [], []
    for d in range(win, T - 1, step):
        S = fn(R[d - win:d]); w = gmv(S)
        lev.append(float(np.abs(w).sum()))
        oos.append(R[d:min(d + step, T)] @ w)
    o = np.concatenate(oos)
    return o.std() * np.sqrt(252) * 100, float(np.mean(lev))


EST_WIN, STEP = 126, 21    # 126 sessions makes q=N/win high enough to separate methods

results = []
for name, fn in ESTIMATORS.items():      # ccc_garch refits GARCH each window (slower)
    vol, lev = backtest(fn, EST_WIN, STEP)
    results.append({"method": name, "oos_vol_pct": round(vol, 2), "gross_leverage": round(lev, 1)})
res = pl.DataFrame(results).sort("oos_vol_pct")
res

In [ ]:
fig = px.bar(res.to_pandas(), x="method", y="oos_vol_pct", color="gross_leverage",
             color_continuous_scale="Reds",
             title=f"Out-of-sample GMV volatility (EST_WIN={EST_WIN})")
fig.update_layout(yaxis_title="annualised vol %", xaxis_title="", height=440)
fig.show()

In [ ]:
# Regime effect: as the estimation window shrinks (q rises), the sample matrix
# degrades while the regularised methods hold. (Fast methods only.)
grid = []
for win in (252, 126, 90):
    for name, fn in FAST.items():
        vol, _ = backtest(fn, win, STEP)
        grid.append({"est_win": win, "q": round(N / win, 2), "method": name,
                     "oos_vol_pct": round(vol, 2)})
pl.DataFrame(grid).pivot(values="oos_vol_pct", index="method", on="est_win").sort("method")

## DCC: time-varying correlation

The constant-correlation model above fixes the correlation matrix. DCC (Engle)
lets it move: the correlation of the standardised residuals follows its own
scalar recursion. Estimated by quasi-MLE on a subset here (the likelihood inverts
an NxN matrix at every step, so cost grows fast with N). The plot shows average
pairwise correlation rising in stress and falling in calm - the dynamic a
constant model cannot capture.

In [ ]:
N_DCC = min(60, N)
Zc = np.column_stack(Parallel(n_jobs=-1)(delayed(_fit_col)(R[:, j]) for j in range(N_DCC)))
Zc = Zc / Zc.std(0)
Qbar = LedoitWolf().fit(Zc).covariance_
dq = np.sqrt(np.diag(Qbar)); Qbar = Qbar / np.outer(dq, dq)
iu = np.triu_indices(N_DCC, 1)


def dcc_nll(p):
    a, b = p
    if a <= 0 or b <= 0 or a + b >= 0.999:
        return 1e12
    Q = Qbar.copy(); nll = 0.0
    for t in range(len(Zc)):
        dd = np.sqrt(np.diag(Q)); Rt = Q / np.outer(dd, dd)
        _, logdet = np.linalg.slogdet(Rt)
        nll += logdet + Zc[t] @ np.linalg.solve(Rt, Zc[t])
        z = Zc[t]; Q = (1 - a - b) * Qbar + a * np.outer(z, z) + b * Q
    return nll


fit = minimize(dcc_nll, [0.02, 0.95], method="Nelder-Mead",
               options={"xatol": 1e-3, "fatol": 1e-1, "maxiter": 60})
a, b = fit.x
print(f"DCC a={a:.3f}  b={b:.3f}  persistence={a + b:.3f}  (N={N_DCC})")

Q = Qbar.copy(); avg_corr = []
for t in range(len(Zc)):
    dd = np.sqrt(np.diag(Q)); Rt = Q / np.outer(dd, dd); avg_corr.append(Rt[iu].mean())
    z = Zc[t]; Q = (1 - a - b) * Qbar + a * np.outer(z, z) + b * Q

dates = px_wide["date"].to_numpy()[1:]
fig = go.Figure()
fig.add_trace(go.Scatter(x=dates, y=avg_corr, mode="lines", name="DCC (dynamic)",
                         line=dict(color="#cf222e", width=1)))
fig.add_hline(y=float(np.mean(avg_corr)), line_dash="dash", line_color="#1f6feb",
              annotation_text="constant-correlation level")
fig.update_layout(title=f"Average pairwise correlation, {N_DCC} names",
                  yaxis_title="mean correlation", height=400)
fig.show()

## Takeaways

- **Sample** is only fine when T is far larger than N; as `EST_WIN` shrinks its
  GMV blows up on leverage and out-of-sample risk.
- **Linear shrinkage (LW / OAS)** is the robust statistical default: no factor
  assumptions, well-conditioned, strong out-of-sample. One intensity for the
  whole spectrum, so the shrinkage curve is a straight line.
- **Nonlinear shrinkage (QIS)** cleans each eigenvalue by its own amount and is
  the optimal estimator that keeps the sample eigenvectors. The shrinkage curve
  shows why: it keeps the signal eigenvalues and lifts only the noise. It matches
  or beats linear shrinkage when T>N and stays positive-definite and competitive
  when N>T, so it is the strongest general-purpose statistical estimator here.
- **PCA and RMT clipping** encode the real factor structure and match or beat
  shrinkage; RMT is the correlation-cleaning view of the same idea.
- **EWMA** is a reactive risk-measurement tool, not an optimiser input at this N
  (rank-deficient, so its GMV is unstable).
- **CCC / DCC GARCH** give a conditional covariance from the univariate GARCH
  work; DCC adds moving correlation. Best when volatility and correlation timing
  matter.

No characteristics data yet, so a fundamental (Barra-style) factor model is not
built here; PCA is its statistical stand-in until ticker details land (0013).